In [1]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .master('local[2]') \
    .appName('zoomcamp-hw') \
    .config('spark.driver.memory', '800m') \
    .config('spark.executor.memory', '800m') \
    .config('spark.driver.memoryOverhead', '128m') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.ui.enabled', 'true') \
    .getOrCreate()

In [56]:
spark.version

'4.0.0'

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

In [ ]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [ ]:
import os
size = os.path.getsize('yellow_tripdata_2025-11.parquet') / (1024*1024)
print(f'Размер файла: {size:.1f} MB')

# Сколько строк и партиций сейчас
print(f'Текущих партиций: {df.rdd.getNumPartitions()}')

In [ ]:
df.show

### Repartition

In [ ]:
df.repartition(4).write.mode('overwrite').parquet('yellow_2025_11_partitioned')

In [ ]:
df.coalesce(4).write.mode('overwrite').parquet('yellow_2025_11_partitioned')

In [2]:
import os
os.environ['SPARK_LOCAL_DIRS'] = '/tmp'

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master('local[1]') \
    .appName('zoomcamp-hw') \
    .config('spark.driver.memory', '512m') \
    .config('spark.driver.memoryOverhead', '256m') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.ui.enabled', 'false') \
    .config('spark.sql.files.maxPartitionBytes', '16m') \
    .config('spark.memory.fraction', '0.6') \
    .config('spark.memory.storageFraction', '0.3') \
    .getOrCreate()

print('Spark started:', spark.version)

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-U

Spark started: 4.0.0


In [3]:
# Читаем с явным указанием mergeSchema=False — быстрее
df = spark.read \
    .option('mergeSchema', 'false') \
    .parquet('yellow_tripdata_2025-11.parquet')

print(f'Партиций при чтении: {df.rdd.getNumPartitions()}')
print(f'Колонок: {len(df.columns)}')

Партиций при чтении: 5
Колонок: 20


In [4]:
df.repartition(4) \
  .write \
  .mode('overwrite') \
  .parquet('yellow_2025_11_partitioned')

print('Готово!')

[Stage 3:============================================>              (3 + 1) / 4]

Готово!


### Schema

In [6]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



### Question 3. Count records

In [ ]:
df.filter((df.tpep_pickup_datetime >= '2025-11-15 00:00:00') & (df.tpep_pickup_datetime <= '2025-11-15 23:59:59')).show()

In [21]:
df.filter((df.tpep_pickup_datetime >= '2025-11-15 00:00:00') & (df.tpep_pickup_datetime <= '2025-11-15 23:59:59')).count()

162604

In [26]:
from pyspark.sql.functions import to_date
df.filter(to_date(df.tpep_pickup_datetime) == '2025-11-15').count()

162604

In [28]:
from pyspark.sql import functions as F
df.filter(F.to_date(df.tpep_pickup_datetime) == '2025-11-15').count()

162604

### Question 4: Longest trip

In [37]:
df \
    .withColumn('Pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .withColumn('Dropoff_date', F.to_date(df.tpep_dropoff_datetime)) \
    .withColumn('Length_hours', (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600) \
    .select('Pickup_date', 'Dropoff_date', 'Length_hours') \
    .show()

+-----------+------------+--------------------+
|Pickup_date|Dropoff_date|        Length_hours|
+-----------+------------+--------------------+
| 2025-11-01|  2025-11-01|                 0.0|
| 2025-11-01|  2025-11-01| 0.20416666666666666|
| 2025-11-01|  2025-11-01| 0.22277777777777777|
| 2025-11-01|  2025-11-01|              1.0175|
| 2025-11-01|  2025-11-01|  0.5116666666666667|
| 2025-11-01|  2025-11-01| 0.17444444444444446|
| 2025-11-01|  2025-11-01|  0.3036111111111111|
| 2025-11-01|  2025-11-01|              0.8675|
| 2025-11-01|  2025-11-01|               0.085|
| 2025-11-01|  2025-11-01| 0.47833333333333333|
| 2025-11-01|  2025-11-01|  0.7061111111111111|
| 2025-11-01|  2025-11-01| 0.21472222222222223|
| 2025-11-01|  2025-11-01|  0.5761111111111111|
| 2025-11-01|  2025-11-01|  0.5083333333333333|
| 2025-11-01|  2025-11-01|  0.8694444444444445|
| 2025-11-01|  2025-11-01|  0.0961111111111111|
| 2025-11-01|  2025-11-01|0.058333333333333334|
| 2025-11-01|  2025-11-01| 0.42833333333

In [38]:
df \
    .withColumn('Length_hours', (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600) \
    .agg(F.max('Length_hours')).collect()[0][0]

90.64666666666666

### Question 6: Least frequent pickup location zone

In [39]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 02:35:15--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
13.33.232.77, 13.33.232.163, 13.33.232.217, ...vzurychx.cloudfront.net)... 
connected. to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.33.232.77|:443... 
200 OKequest sent, awaiting response... 
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.04s   

2026-03-08 02:35:15 (271 KB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [44]:
df_zones = spark.read.csv('taxi_zone_lookup.csv', header=True)

In [45]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [47]:
df_zones.registerTempTable('zones')

In [48]:
df.registerTempTable('trips')

In [53]:
spark.sql("""
    SELECT 
        zones.Zone,
        COUNT(*)
    FROM trips
    LEFT JOIN zones ON trips.PULocationID = zones.LocationID
    GROUP BY zones.Zone
    ORDER BY COUNT(*)
""").show(truncate=False)

+---------------------------------------------+--------+
|Zone                                         |count(1)|
+---------------------------------------------+--------+
|Governor's Island/Ellis Island/Liberty Island|1       |
|Eltingville/Annadale/Prince's Bay            |1       |
|Arden Heights                                |1       |
|Port Richmond                                |3       |
|Rikers Island                                |4       |
|Rossville/Woodrow                            |4       |
|Green-Wood Cemetery                          |4       |
|Great Kills                                  |4       |
|Jamaica Bay                                  |5       |
|Westerleigh                                  |12      |
|Crotona Park                                 |14      |
|Oakwood                                      |14      |
|New Dorp/Midland Beach                       |14      |
|West Brighton                                |14      |
|Willets Point                 